In [1]:
import pandas as pd
import os
import re

raw_folder = r"./data/"
all_years = []

for file in os.listdir(raw_folder):
    if not file.lower().endswith((".xls", ".xlsx")):
        continue

    file_path = os.path.join(raw_folder, file)

    # Extract fiscal year (use ending year)
    match = re.search(r"(\d{4})-(\d{4})", file)
    fiscal_year = match.group(2) if match else None

    try:
        # Choose engine by extension
        if file.lower().endswith(".xls"):
            xls = pd.ExcelFile(file_path, engine="xlrd")
        else:
            xls = pd.ExcelFile(file_path, engine="openpyxl")

        # Find the ALC sheet safely
        alc_sheet = [s for s in xls.sheet_names if "ALC" in s][0]

        df = pd.read_excel(
            xls,
            sheet_name=alc_sheet,
            skiprows=4
        )

        df.columns = [
            "Province/territory",
            "Number of hospitalizations",
            "Number of hospitalizations with reported ALC days",
            "Proportion of hospitalizations with reported ALC days (%)",
            "Length of stay (in days) Total LOS",
            "Length of stay (in days) ALC LOS",
            "Patient days in ALC (%)"
        ]

        df.insert(0, "Fiscal Year", fiscal_year)

        df = df[df["Province/territory"].notna()]
        df = df[~df["Province/territory"].str.contains("Note", na=False)]

        all_years.append(df)

        print(f"✅ Processed: {file}")

    except Exception as e:
        print(f"⚠️ Skipped {file}: {e}")

final_df = pd.concat(all_years, ignore_index=True)
final_df.to_csv("Hospitalization numbers.csv", index=False)

print("✅ Export complete: Hospitalization numbers.csv")

✅ Processed: Hospitalization rates & others 2017-2018.xlsx
⚠️ Skipped Hospitalization rates & others 2018-2019.xlsx: Length mismatch: Expected axis has 13 elements, new values have 7 elements
⚠️ Skipped Hospitalization rates & others 2019-2020.xlsx: Length mismatch: Expected axis has 13 elements, new values have 7 elements
⚠️ Skipped Hospitalization rates & others 2020-2021.xlsx: Length mismatch: Expected axis has 13 elements, new values have 7 elements
⚠️ Skipped Hospitalization rates & others 2021-2022.xlsx: Length mismatch: Expected axis has 13 elements, new values have 7 elements
⚠️ Skipped Hospitalization rates & others 2022-2023.xlsx: Length mismatch: Expected axis has 13 elements, new values have 7 elements
⚠️ Skipped Hospitalization rates & others 2023-2024.xlsx: list index out of range
⚠️ Skipped Hospitalization rates & others 2024-2025.xlsx: list index out of range
✅ Processed: Master_Hospitalization rates & others 2017-2025.xlsx
⚠️ Skipped ~$Hospitalization rates & others 20